In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Aumenta o tamanho das figuras (gráficos)
plt.rcParams['figure.dpi'] = 150
# Aumenta a largura do JupyterNotebook
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:65% !important; }</style>"))

In [2]:
# DataFrame com os incidentes
df = pd.read_feather("incidents_final_enriched_LIMPO.feather")

# Disclaimer
<br>

#### Essas informações não foram trabalhadas detalhadamente devido ao tempo de entrega exigido, onde a maior parte da análise foi feita na base de incidentes, sendo assim esse Notebook tem caráter objetivo, de extrair dados essenciais para prosseguir com a análise.

## Categorias de Custos para Incidentes
<br>

### 1. Custos de Downtime/Parada Operacional
**Descrição**: Incidentes relacionados a interrupções, paradas emergenciais, falhas de sistemas críticos que param ou reduzem a produção.

**Exemplos de incidentes**:
- `SSM - Interrupção não programada superior a 24 horas`
- `SSM - Parada emergencial de planta de processo`
- `SSM - Falha de sistema crítico de segurança operacional`
- `SSM - Falha na demanda total ou parcial de sistema crítico`

**Custos associados**:
- Perda de produção (horas/dias parados × valor produção/hora)
- Custos de remobilização de equipes
- Horas extras para retomada
- Penalidades por atraso em contratos

---

### 2. Custos Ambientais (Multas/Remediação)
**Descrição**: Incidentes com descargas, vazamentos, contaminações que geram impacto ambiental e exigem ações corretivas.

**Exemplos de incidentes**:
- `SSM - Descarga de óleo/água/produtos químicos` (maior/significativa/menor)
- `SSM - Perda de contenção` primária de fluidos
- `SSM - Descarte fora de especificação`
- `SSM - Vazamento de H2S`
- `SSM - Constatação de mancha de origem indeterminada`

**Custos associados**:
- Multas ambientais (Ibama, órgãos estaduais)
- Custos de limpeza/remoção
- Monitoramento ambiental pós-evento
- Compensações ambientais
- Custos de disposição de resíduos

---

### 3. Custos de Saúde/Segurança
**Descrição**: Incidentes com danos corporais, fatalidades, doenças ocupacionais que afetam pessoas.

**Exemplos de incidentes**:
- `SSM - Fatalidade`
- `SSM - Ferimento grave` / com afastamento
- `SSM - Homem ao mar`
- `SSM - Surto de doença infectocontagiosa`
- `SSM - Quase acidente de alto potencial`

**Custos associados**:
- Custos médicos/hospitalares
- Indenizações trabalhistas
- Perda de produtividade por afastamento
- Custos de investigação/acompanhamento
- Aumento de prêmios de seguro

---

### 4. Custos de Danos a Equipamentos/Ativos
**Descrição**: Incidentes que danificam ativos físicos, estruturas, equipamentos exigindo reparos ou substituição.

**Exemplos de incidentes**:
- `SSM - Falha estrutural` (offshore, tanque, poço)
- `SSM - Explosão mecânica`
- `SSM - Abalroamento significante`
- `SSM - Queda de objetos/equipamentos`
- `SSM - Falha no sistema de ancoragem`

**Custos associados**:
- Reparo/substituição de equipamentos
- Custos de engenharia/projeto
- Logística especializada (offshore)
- Perda de ativo (se irreparável)
- Depreciação acelerada

---

### 5. Custos de Perda de Controle Operacional
**Descrição**: Incidentes relacionados a perda de controle de processos críticos que podem evoluir para emergências catastróficas.

**Exemplos de incidentes**:
- `SSM - Perda de controle de poço` (maior/significativa/menor)
- `SSM - Falha da barreira primária (kick)`
- `SSM - Falha no Blowout Preventer (BOP)`
- `SSM - Perda de contenção de H2S`
- `SSM - Detonação acidental de explosivos`

**Custos associados**:
- Resposta emergencial em larga escala
- Equipes especializadas de contenção
- Potencial blowout (jorro descontrolado)
- Investigação técnica complexa
- Revisão completa de procedimentos
- Paralisação prolongada da unidade

## 1. Custos de Downtime/Parada Operacional
<br>

#### Foi utilizado documento referênte ao primeiro trimestre de 2025 da Petrobras para estipular o valor perdido por interrompimento da produção. Informação retirada de: https://www.investidorpetrobras.com.br/resultados-e-comunicados/central-de-resultados/
<br>

#### Caminho do arquivo: Release de Resultados em R$ -> 1T25:
<br>

#### Produção total diária da Petrobras (óleo + gás): 2,77 milhões de barris por dia (página 5)
#### Lucro operacional do segmento E&P no trimestre: R$ 44,168 bilhões (Tabela 7)

#### O calculo foi feito usando esse valor trimestral por 90 dias e depois dividido entre os valores únicos de instalações do CNPJ da Petrobras

In [3]:
Quantidade_barris_dia = 2770000
lucro_operacional_trimestral = 44168000000
lucro_operacional_diario = lucro_operacional_trimestral / 90 # Lucro diário somando todas unidades
lucro_operacional_diario

print(f'Lucro operacional diario: {lucro_operacional_diario}\n')

quantidade = df['CNPJ'].astype(str).str.contains('33000167000101').sum()
print(f'Quantidade de CNPJs da Petrobras: {quantidade}\n')

instalacoes_unicas = df.loc[df['CNPJ'].astype(str).str.contains('33000167000101'), 'Instalacao'].dropna().nunique()
print(f'Quantidade de instalações unicas: {instalacoes_unicas}\n')

valor_diluido_instalações = lucro_operacional_diario / instalacoes_unicas
print(f'Valor do lucro diario distribuido pela quantidade de instalações: {valor_diluido_instalações}')


Lucro operacional diario: 490755555.5555556

Quantidade de CNPJs da Petrobras: 22024

Quantidade de instalações unicas: 3278

Valor do lucro diario distribuido pela quantidade de instalações: 149711.88394007186


In [4]:
Quantidade_barris_dia = 2_770_000  # Usando underscore para legibilidade
lucro_operacional_trimestral = 44_168_000_000  # 44,168 bilhões

# Lucro diário (considerando 90 dias no trimestre)
lucro_operacional_diario = lucro_operacional_trimestral / 90

# Contagens do DataFrame
quantidade_cnpj = df['CNPJ'].astype(str).str.contains('33000167000101').sum()
instalacoes_unicas = df.loc[df['CNPJ'].astype(str).str.contains('33000167000101'), 'Instalacao'].dropna().nunique()

# Valor diluído por instalação
valor_diluido_instalacoes = lucro_operacional_diario / instalacoes_unicas

# Função para formatar números grandes
def formatar_numero(numero):
    if numero >= 1_000_000_000:
        return f"R$ {numero/1_000_000_000:,.2f} bilhões"
    elif numero >= 1_000_000:
        return f"R$ {numero/1_000_000:,.2f} milhões"
    elif numero >= 1_000:
        return f"R$ {numero/1_000:,.2f} mil"
    else:
        return f"R$ {numero:,.2f}"

# Função para formatar números inteiros com separador de milhar
def formatar_inteiro(numero):
    return f"{numero:,.0f}".replace(",", ".")

# Resultados formatados
print("=" * 60)
print("RESULTADOS DA ANÁLISE")
print("=" * 60)
print(f"\n Quantidade de barris/dia: {formatar_inteiro(Quantidade_barris_dia)}")
print(f" Lucro operacional trimestral: {formatar_numero(lucro_operacional_trimestral)}")
print(f" Lucro operacional diário: {formatar_numero(lucro_operacional_diario)}")
print(f" Quantidade de CNPJs da Petrobras: {formatar_inteiro(quantidade_cnpj)}")
print(f" Quantidade de instalações únicas: {formatar_inteiro(instalacoes_unicas)}")
print(f" Valor do lucro diário por instalação: {formatar_numero(valor_diluido_instalacoes)}")
print("\n" + "=" * 60)

RESULTADOS DA ANÁLISE

 Quantidade de barris/dia: 2.770.000
 Lucro operacional trimestral: R$ 44.17 bilhões
 Lucro operacional diário: R$ 490.76 milhões
 Quantidade de CNPJs da Petrobras: 22.024
 Quantidade de instalações únicas: 3.278
 Valor do lucro diário por instalação: R$ 149.71 mil



#### Leve - 1 dia
#### Moderado - 9 dias
#### Grave - 30 dias
<br>

##### OGP 418 – Equipment Reliability and Maintenance Data for Offshore Operations: https://www.iogp.org/bookstore/
##### ABS Guidance Notes on Reliability-Centered Maintenance:  https://ww2.eagle.org/content/dam/eagle/publications/retrofit-technologies.pdf

### Resultado de R$ 149.71 mil para cada dia de Downtime

## ------------------------------------------------------------------------------------------------------------------------

## 2. Custos Ambientais (Multas/Remediação)
<br>

#### CLASSIFICAÇÃO DE PREJUÍZO AMBIENTAL (BASE: PORTARIA IBAMA Nº 115/2022):
https://www.ibama.gov.br/component/legislacao/?view=legislacao&legislacao=139170

## RESUMO GERAL (MÉDIA COMO ESTIMADOR)
#### Categoria	Faixa Legal (Portaria 115/2022)	Média adotada (custo estimado)
<br>

    Leve     RS 1.000 – RS 5.000	        RS 3.000
    Moderado    RS 5.000 – RS 500.000	RS 252.500
    Grave    RS 50.000 – RS 50.000.000	RS 25.025.000

### Dados de multas do IBAMA SP

In [5]:
# Carregando base de dados de multas pelo IBAMA
df_multa_sp = pd.read_csv('multasDistribuidasBensTutelados.csv', delimiter=';')

# Filtra onde 'Nome ou Razão Social' contém PETROBRAS
df_multa_sp_petrobras = df_multa_sp[
    df_multa_sp['Nome ou Razão Social'].str.contains('PETROBRAS', case=False, na=False)
].copy()
df_multa_sp_petrobras

,Nº AI,Data Auto,Nome ou Razão Social,CPF/CNPJ,UF,Município,Tipo Auto,Tipo Infração,Enquadramento Legal,Valor do Auto,Moeda,Situação Débito,Última Atualização Relatório
4,00I3N3ZK -,12/03/2025,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,SAO PAULO,Multa,Controle ambiental,Lei 9966/00 - Artigo 27,50.000,Real,Para homologação/prazo de defesa,01/12/2025 22:05
25,075FP6VL -,12/03/2025,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,SAO PAULO,Multa,Controle ambiental,Lei 9966/00 - Artigo 27,26.000,Real,Para homologação/prazo de defesa,01/12/2025 22:05
62,0FAVZBHH -,13/03/2025,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,SAO PAULO,Multa,Controle ambiental,Lei 9966/00 - Artigo 27,26.000,Real,Para homologação/prazo de defesa,01/12/2025 22:05
97,0NMU8RJ0 -,10/10/2022,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,SAO PAULO,Multa DiÃ¡ria,Controle ambiental,Decreto 6514/2008 - Artigo 81,130,Real,Quitado. Baixa automática,01/12/2025 22:05
5238,1UD5FWL8 -,20/10/2023,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,ILHABELA,Multa,Controle ambiental,"Decreto 6514/2008 - Artigo 62, Decreto 6514/20...",5.000,Real,Quitado. Baixa automática,01/12/2025 22:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39160,YYLNXHRR -,27/12/2023,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,SAO PAULO,Multa,Controle ambiental,Lei 9966/00 - Artigo 27,150.000,Real,Quitado. Baixa automática,01/12/2025 22:05
39196,Z676OSHF -,12/05/2023,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,ILHABELA,Multa,Controle ambiental,Decreto 6514/2008 - Artigo 62,30.000,Real,Para homologação/prazo de defesa,01/12/2025 22:05
39250,ZK0MQYLG -,10/05/2023,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,ILHABELA,Multa,Controle ambiental,Lei 9966/00 - Artigo 27,26.000,Real,Quitado. Baixa automática,01/12/2025 22:05
39283,ZQ8AA538 -,18/12/2020,PETROLEO BRASILEIRO S/A - PETROBRAS,33.000.167/0001-01,SP,SANTOS,Multa,Controle ambiental,Decreto 6514/2008 - Artigo 62,30.000,Real,Para homologação/prazo de defesa,01/12/2025 22:05


#### Transformando Valor do Auto para fazer contas

In [6]:
df_multa_sp_petrobras['Valor do Auto'] = (
    df_multa_sp_petrobras['Valor do Auto']
    .astype(str)
    .str.replace('.', '', regex=False)  # Remove TODOS os pontos
    .astype(float)                       # Agora converte para float para poder fazer contas
)

#### Verificando as médias com e sem outliers

In [7]:
# Calcula percentis 5% e 95%
p5 = df_multa_sp_petrobras['Valor do Auto'].quantile(0.05)
p95 = df_multa_sp_petrobras['Valor do Auto'].quantile(0.95)

# Filtra valores entre percentis 5 e 95
valores_centrais = df_multa_sp_petrobras['Valor do Auto'][
    (df_multa_sp_petrobras['Valor do Auto'] >= p5) & 
    (df_multa_sp_petrobras['Valor do Auto'] <= p95)
]

# Médias
media_completa = df_multa_sp_petrobras['Valor do Auto'].mean()
media_90centrais = valores_centrais.mean()

print(f"Média completa: R$ {media_completa:,.2f}")
print(f"Média sem 5% extremos: R$ {media_90centrais:,.2f}")

Média completa: R$ 68,759.41
Média sem 5% extremos: R$ 34,289.45


#### O valor das médias foi calculado para verificar o grau de confiabilidade das faixas passadas pelo IBAMA em sua faixa legal, apesar de variar muito.

## ------------------------------------------------------------------------------------------------------------------------

## 3. Custos de Saúde/Segurança
<br>

#### TEXTO FALANDO SOBRE



| Severidade | Faixa de Custo (R$) | Justificativa | Referências |
|------------|---------------------|---------------|-------------|
| **LEVE** | 500 – 5.000 | A literatura indica que eventos de baixa gravidade envolvem custos diretos como primeiros socorros, atendimentos ambulatoriais, medicação simples e pequenas interrupções operacionais. Estudos do MTE e da OIT mostram que esses custos diretos são relativamente baixos quando não há afastamento significativo. | • OIT – Custos de Acidentes de Trabalho (custos diretos baixos em ocorrências sem afastamento): https://www.ilo.org<br>• MTE – Acidentes sem afastamento e seus impactos diretos: https://www.gov.br/trabalho |
| **MODERADO** | 5.000 – 50.000 | Casos com lesões moderadas apresentam custos maiores devido a afastamentos prolongados, exames diagnósticos, tratamento clínico, substituição temporária de pessoal e perda parcial de produtividade. Pesquisas amplamente utilizadas no setor indicam que acidentes com afastamento elevam rapidamente o custo total. | • ANAMT – Custos do adoecimento e afastamento: https://www.anamt.org.br<br>• OSHA (EUA) – Indirect vs. direct accident costs: https://www.osha.gov |
| **GRAVE** | 50.000 – 500.000+ | A literatura mostra que acidentes graves envolvem internações, cirurgias, longas reabilitações, indenizações, paralisação de operações e custos legais. Estudos internacionais e dados do MPT apontam que acidentes graves aumentam os custos totais em múltiplos. | • MPT – Custos de acidentes graves e fatais: https://www.mpt.mp.br<br>• OSHA – Severe injury cost factors: https://www.osha.gov |

    Leve     RS 500 – RS 5.000	        RS 2.250
    Moderado    RS 5.000 – RS 500.000	RS 249.750
    Grave    RS 50.000 – RS 50.000.000	RS 25.025.000

## ------------------------------------------------------------------------------------------------------------------------

## 4. Custos de Danos a Equipamentos/Ativos
<br>

#### TEXTO FALANDO SOBRE

| Severidade do Dano | Faixa de Custo (R$) | Justificativa | Referências |
| :--- | :--- | :--- | :--- |
| **Baixo (Falha Menor)** | 10.000 – 100.000 | Falha de componentes não críticos, reparo local, substituição de peças de baixo valor, sem interrupção significativa da produção. Custos primários de mão de obra e peças. | **Estudo de Caso - Manutenção Preditiva (Ocyan):** [https://todaia.com.br/oleo-e-gas-manutencao-preditiva-de-plataformas-offshore-com-ia-o-estudo-de-caso-da-ocyan/](https://todaia.com.br/oleo-e-gas-manutencao-preditiva-de-plataformas-offshore-com-ia-o-estudo-de-caso-da-ocyan/ ) |
| **Médio (Falha Crítica)** | 100.000 – 5.000.000 | Falha de equipamentos críticos (bombas, válvulas, pequenos trechos de tubulação), exigindo reparo complexo ou substituição. Envolve custos de mobilização de equipes especializadas e potencial interrupção de curto prazo. | **Dissertação - Corrosão Offshore:** [https://repositorio.unifesp.br/items/c9142eb7-047c-40cf-874f-866b41609542](https://repositorio.unifesp.br/items/c9142eb7-047c-40cf-874f-866b41609542 ) |
| **Alto (Falha Catastrófica)** | 5.000.000 – 50.000.000+ | Falha estrutural, dano a equipamentos de alto valor (ANM, compressores principais) ou perda de integridade de barreira. Envolve custos de substituição de ativos multimilionários e longos períodos de *downtime*. | **Estudo de Caso - Falha Submarina:** [https://ojs.brazilianjournals.com.br/ojs/index.php/BRJD/article/download/60145/43472](https://ojs.brazilianjournals.com.br/ojs/index.php/BRJD/article/download/60145/43472 ) |


 #### Médias
    Leve RS45.000,00
    Moderado RS2.250.000,00
    Grave RS22.500.000,00

## ------------------------------------------------------------------------------------------------------------------------

## 5. Custos de Perda de Controle Operacional
<br>

#### TEXTO FALANDO SOBRE

| Severidade do Incidente | Faixa de Custo (R$) | Justificativa | Referências |
| :--- | :--- | :--- | :--- |
| **Baixo (Falha de Barreira)** | 50.000 – 500.000 | Custos de investigação interna, reparos imediatos e multas mínimas por falhas de segurança de processo. | Relatório SEBRAE/Baktron: https://baktron.com.br/wp-content/uploads/2019/06/20171005-RI-PG-ServAmb_v4.pdf |
| **Médio (Perda de Contenção/Posicionamento)** | 500.000 – 5.000.000 | Multas regulatórias e custos operacionais decorrentes de infrações de segurança. | Lei nº 9.847/1999: http://www.planalto.gov.br/ccivil_03/leis/l9847.htm |
| **Alto (Acidente de Processo Catastrófico)** | 5.000.000 – 50.000.000+ | Incidentes com múltiplas penalidades, remediação extensa e perda de imagem. | CCPS/AIChE Process Safety Metrics: https://www.aiche.org/sites/default/files/docs/pages/ccps_process_safety_metrics_-_v3.1_-_pt_final.pdf |


#### Médias
    Leve RS225.000,00
    Moderado RS 2.250.000,00
    Grave RS22.500.000,00

### REVISAR LINKS SAUDE